# LLM mapping — AST pattern → organism / resistance group

This notebook contains **only** the pipeline that:

1. Loads the ICU microbiology AST extract (`all_micro_icu_df`).
2. Builds one text row per culture (`stay_id`, `org_name`, `storetime`) as **`AST_PATTERN`**.
3. Exports a slim table for the **external LLM** (`index`, `org_name`, `AST_PATTERN`).
4. Merges back **`mapped_letter`** (single letter A–M per thesis taxonomy).
5. Maps letters → **`final_class`** and saves **`mapper_ab_org_llm_df`** for downstream merges.

**Prerequisite:** run the main ETL that writes `all_micro_df_*.parquet`, or point `MICRO_PARQUET` to your file.

In [ ]:
import pandas as pd
from pathlib import Path

import tools.helpers as hh

DATA_ROOT = Path("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building")
PARQ_DIR = DATA_ROOT / "parq"
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Same extract as in 02_03_Feature_space.ipynb — adjust if you regenerate micro data
MICRO_PARQUET = PARQ_DIR / "all_micro_df_26Feb26_0625.parquet"

## 1. Load AST-level microbiology table

In [ ]:
all_micro_icu_df = hh.load_data(str(MICRO_PARQUET))
hh.dx(all_micro_icu_df)

## 2. Build `AST_PATTERN` (concatenated drug–bug–interpretation strings per culture)

Each row in `all_ast_df` is one antibiotic–organism interpretation. Rows are rolled up to one string per (`stay_id`, `org_name`, `storetime`).

In [ ]:
all_micro_icu_df["combined_col"] = all_micro_icu_df[
    ["org_name", "ab_name", "interpretation"]
].values.tolist()
all_micro_icu_df["combined_col"] = all_micro_icu_df["combined_col"].astype(str)

all_ast_df = all_micro_icu_df.dropna(
    subset=["org_name", "ab_name", "interpretation"]
)

mapper_df = (
    all_ast_df.groupby(["stay_id", "org_name", "storetime"], sort=False)["combined_col"]
    .apply(lambda x: "".join(x))
    .reset_index(name="AST_PATTERN")
)
mapper_df = mapper_df.reset_index()
mapper_df.head()

## 3. Export payload for external LLM

Send `mapper_dff` to your LLM. Preserve **`index`** so results can be joined back row-for-row.

In [ ]:
mapper_dff = mapper_df[["index", "org_name", "AST_PATTERN"]].copy()
export_path = OUTPUT_DIR / "mapper_dff_for_llm.json"
mapper_dff.to_json(export_path, orient="records", lines=True)
print(f"Wrote {len(mapper_dff)} rows → {export_path}")
mapper_dff.head()

## 4. Merge LLM output (`mapped_letter`)

After the external run, provide a table with **`index`** and **`mapped_letter`** (one character A–M per row).

- **Option A:** load a JSON/CSV you saved from the LLM.
- **Option B:** load an already-merged parquet from a previous run (e.g. `mapper_ab_org_llm_df_*.parquet`) and skip to section 5.

In [ ]:
# --- Option A: merge fresh LLM output ---
# llm_path = OUTPUT_DIR / "mapper_llm_mapped_letters.json"  # or .csv
# llm_letters = pd.read_json(llm_path)  # columns: index, mapped_letter
# mapper_df = mapper_df.merge(
#     llm_letters[["index", "mapped_letter"]], on="index", how="left"
# )
# assert mapper_df["mapped_letter"].notna().all(), "Some rows missing LLM letter"

# --- Option B: reload completed artifact (uncomment path you use) ---
mapper_df = hh.load_data(str(PARQ_DIR / "mapper_ab_org_llm_df_27Feb26_1644.parquet"))
hh.dx(mapper_df, pid_col="stay_id")

## 5. Letter → `final_class` (13 thesis groups)

In [ ]:
class_code_mapper = {
    "AmpC_Producers": "A",
    "Non_ESBL_Enterobacterales": "B",
    "Enterococcus_VRE": "C",
    "ESBL_Enterobacterales": "D",
    "MRSA": "E",
    "Low_Significance": "F",
    "Enterococcus_VSE": "G",
    "Other_NonFermenters": "H",
    "MSSA": "I",
    "Streptococcus_pneumoniae": "J",
    "Acinetobacter": "K",
    "Pseudomonas": "L",
    "Carbopenam Resistant Enterobacterales": "M",
}

reverse_class_code_mapper = {v: k for k, v in class_code_mapper.items()}
mapper_df["final_class"] = mapper_df["mapped_letter"].map(reverse_class_code_mapper)
mapper_df[["index", "stay_id", "org_name", "mapped_letter", "final_class"]].head()

## 6. Save labeled table for feature notebooks

Downstream: merge on (`stay_id`, `org_name`, `storetime`) or use `index` if stable across runs.

In [ ]:
hh.parq(mapper_df, "mapper_ab_org_llm_df_")